# 00 — Production Traffic + Monitoring

Run once before `01_eval_and_promote` / `02_human_review`. Environment prep (reset + seed-data loading) lives in `src/prep.py`; this notebook focuses on the eval/monitoring flow:
1. Install dependencies
2. Prepare the environment — one call to `src/prep.py` (optional reset + load seed tables)
3. Register the baseline prompt (v1) under the `production` alias
4. Set the experiment + start production monitoring (5 scorers)
5. Send 80 tickets through the agent as production traffic (scored async by monitoring)

## 1. Install dependencies

Skip the restart cell if your cluster already has these.

In [ ]:
%pip install -U mlflow databricks-sdk openai python-dotenv databricks-agents

In [ ]:
dbutils.library.restartPython()

## 2. Imports and config

Adds the repo's `src/` directory to the Python path so the agent, tools, scorers, and data-gen modules are importable.

In [ ]:
import os
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT))

import mlflow
from src import config
from src.agent import run_agent
from src.prep import prepare

print(f'Workspace: {config.DATABRICKS_HOST}')
print(f'Prompt name: {config.PROMPT_NAME}')
print(f'Agent model: {config.AGENT_MODEL}')
print(f'Judge model: {config.JUDGE_MODEL}')

## 3. Prepare the environment

One call to `src/prep.py` handles the boilerplate: optional **reset** (drop tables / prompt / experiment for a clean re-run), ensure the synthetic tickets exist, load all seed JSON into Unity Catalog tables, and return the tickets. Set `RESET=False` to keep existing state.

In [ ]:
RESET = True  # set False to keep existing tables / prompt / experiment
tickets = prepare(spark, reset=RESET)

## 4. Register baseline prompt in MLflow Prompt Registry

We register `system_prompt_v1.md` as version 1 of `your_catalog.case_ticket.support_triage_prompt` and point the `production` alias at it. The agent reads `prompts:/{name}@production` — so any future re-aliasing flips the deployed prompt without touching agent code.

In [ ]:
# Idempotent: only register v1 + set the alias if the prompt isn't already published.
# Keeps a pristine version history across repeated setup runs (no stacked duplicates).
prompt_text = (config.PROMPTS_DIR / 'system_prompt_v1.md').read_text()

try:
    existing = mlflow.genai.load_prompt(f"prompts:/{config.PROMPT_NAME}@{config.PROMPT_ALIAS}")
    prompt_version_num = existing.version
    print(f"Prompt already published at version {prompt_version_num} — skipping registration")
except Exception:
    pv = mlflow.genai.register_prompt(
        name=config.PROMPT_NAME,
        template=prompt_text,
        commit_message='Baseline v1 — deliberately verbose drafts, unprofessional tone, and eager escalation. Demo iteration target.',
        tags={'flaws': 'verbose,unprofessional_tone,eager_escalation', 'iteration': 'baseline'},
    )
    mlflow.genai.set_prompt_alias(
        name=config.PROMPT_NAME,
        alias=config.PROMPT_ALIAS,
        version=pv.version,
    )
    prompt_version_num = pv.version
    print(f"Registered {config.PROMPT_NAME} version {prompt_version_num}; alias '{config.PROMPT_ALIAS}' -> {prompt_version_num}")


## 5. Set up MLflow experiment + production monitoring

Set the experiment, register all 5 scorers, and **start monitoring** (they run on a sample of incoming production traces, async; results in the Scorers tab + monitoring dashboards).

In [ ]:
# Set the experiment, register the 5 scorers, and start production monitoring.
from src.scorers import ALL_SCORERS
from mlflow.genai.scorers import ScorerSamplingConfig
from mlflow.tracing import set_databricks_monitoring_sql_warehouse_id

mlflow.set_experiment(config.MLFLOW_EXPERIMENT_NAME)
EXPERIMENT_ID = mlflow.get_experiment_by_name(config.MLFLOW_EXPERIMENT_NAME).experiment_id
print(f'Experiment: {config.MLFLOW_EXPERIMENT_NAME}')

if config.MONITORING_WAREHOUSE_ID:
    set_databricks_monitoring_sql_warehouse_id(config.MONITORING_WAREHOUSE_ID)
    for _scorer in ALL_SCORERS:
        try:
            _registered = _scorer.register(name=_scorer.name)                          # -> Scorers tab
            _registered.start(sampling_config=ScorerSamplingConfig(sample_rate=1.0))    # begin monitoring
            print('Monitoring scorer:', _scorer.name)
        except Exception as _e:
            print('Scorer monitoring skipped:', getattr(_scorer, 'name', _scorer), '->', _e)
else:
    print('MONITORING_WAREHOUSE_ID not set — skipping scorer monitoring registration')

## 6. Run the agent on 80 tickets (production traffic)

Each `run_agent` call is auto-traced and logged as production traffic — that's what the scheduled monitoring scorers score. `01` / `02` then select from these traces.

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed

# Send each ticket through the agent as production traffic; the scheduled monitoring scorers
# score these traces on a sampled, async schedule. Each trace is tagged environment=production
# (config.PRODUCTION_TRACE_TAGS) so 01/02 can tell production traffic from evaluation runs.
results = []
print(f'Sending {len(tickets)} tickets through the agent as production traffic...')
with ThreadPoolExecutor(max_workers=3) as pool:
    futures = {pool.submit(run_agent, t, trace_tags=config.PRODUCTION_TRACE_TAGS): t['ticket_id'] for t in tickets}
    for i, fut in enumerate(as_completed(futures)):
        tid = futures[fut]
        try:
            results.append({'ticket_id': tid, 'output': fut.result()})
        except Exception as e:
            print(f'  {tid}: FAILED {e}')
        if (i + 1) % 10 == 0:
            print(f'  {i + 1}/{len(tickets)} done')

print(f'Production traffic complete: {len(results)}/{len(tickets)} tickets. Scheduled monitoring will score them.')

## Done

You now have:
- Prompt `support_triage_prompt@production` (v1, the deliberately flawed baseline)
- 80 production-traffic traces in the experiment, which the scheduled monitoring scorers score
- The 5 scorers registered for monitoring (Scorers tab)

Next: **`01_eval_and_promote.ipynb`** and **`02_human_review.ipynb`**.